# Benchmark đối chứng: Qwen2.5-Coder-7B base / instruct (n=5, cùng harness với bản fine-tune)

Mục đích: trả lời *"điểm tăng của bản fine-tune là do fine-tune hay do cách đánh giá?"*. Bản benchmark `bench-mark-peft.ipynb` so FT (n=5, prompt ChatML, 4-bit) với base/instruct (n=1, prompt raw), nên có 3 biến nhiễu: số mẫu, format prompt, và harness. Notebook này chạy **base và instruct với đúng harness của FT** (n=5, timeout 15s, thêm import của prompt, `StopOnCodeBlockEnd`, 4-bit NF4), mỗi model ở cả hai format:

| format | ý nghĩa |
|---|---|
| `raw` | completion thuần (cách eval base ban đầu) |
| `chatml` | bọc `<|im_start|>user ... <|im_end|>\n<|im_start|>assistant\n` không system prompt (giống lúc train FT) |

Kết quả lưu vào `data/results_{key}_{dataset}_few_shot_n5.json` cùng định dạng file FT, nên có thể paired-bootstrap trực tiếp theo `task_id`.

**Thời gian:** ~1.6 giờ/dataset/combo trên T4 với HF generate (4 combo × 2 dataset ≈ 12 giờ). `SKIP_IF_EXISTS=True` cho phép chạy tiếp qua nhiều session; `COMBOS` xếp theo độ ưu tiên.

In [ ]:
# Không đụng torch/torchvision (Kaggle đã ghép sẵn đúng bản). Nếu sau khi cài lại báo lỗi import
# transformers/peft thì Restart Session rồi chạy lại từ cell tiếp theo.
!pip install -q -U peft bitsandbytes accelerate datasets

## 1. Config

In [ ]:
import gc
import math
import multiprocessing
import json
import re
import os
import time
from tqdm import tqdm
import torch
from datasets import load_dataset
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODELS = {
    "qwen_base":     "Qwen/Qwen2.5-Coder-7B",
    "qwen_instruct": "Qwen/Qwen2.5-Coder-7B-Instruct",
}
# (model, prompt_format) theo thứ tự ưu tiên: base/raw đo nhiễu n=1 vs n=5, base/chatml tách riêng hiệu ứng prompt.
COMBOS = [
    ("qwen_base", "raw"),
    ("qwen_base", "chatml"),
    ("qwen_instruct", "raw"),
    ("qwen_instruct", "chatml"),
]
def model_key(m, fmt): return f"{m}_n5ctl_{fmt}"       # vd qwen_base_n5ctl_raw

NUM_SAMPLES    = 5
K_VALUE        = 1
MAX_NEW_TOKENS = 1024
TEMPERATURE    = 0.2
TOP_P          = 0.95

PROMPT_FORMAT  = "raw"              # được cập nhật trong vòng lặp ở mục 6
RUN_TEMPLATES  = ["few_shot"]
LIMIT          = None
SKIP_IF_EXISTS = True

EXEC_TIMEOUT   = 15
RESULT_SUFFIX  = f"_n{NUM_SAMPLES}"

RESULTS_DIR = "/kaggle/working/data"
os.makedirs(RESULTS_DIR, exist_ok=True)

DATASETS = {
    "humaneval":      ("openai/openai_humaneval", "test"),
    "humaneval_plus": ("evalplus/humanevalplus", "test"),
}

# Kết quả FT (n=5, chatml, epoch-1/2/3) để in bảng so sánh ở cuối
FT_RESULTS = {
    "humaneval":      {"ft ep1": 0.7866, "ft ep2": 0.7744, "ft ep3": 0.7598},
    "humaneval_plus": {"ft ep1": 0.7354, "ft ep2": 0.7183, "ft ep3": 0.7061},
}

hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
hf_token = hf_token or os.environ.get("HF_TOKEN")
if hf_token:
    login(hf_token)

## 2. Prompt Templates (giữ nguyên)

In [ ]:
ZERO_SHOT_PROMPT = """You are an expert Python programmer. Write the implementation for the following problem.
Do not output anything else but the code block.

Requirements:
- Output only a single Python code block.
- Include all necessary imports explicitly (e.g., from typing import List, Optional, etc.).
- Do not assume any helper functions exist — define everything you use.
- The solution must be self-contained and executable.

Problem:
{prompt}

Code:
"""

FEW_SHOT_PROMPT = """You are an expert Python programmer. You will be provided some examples, and then a problem to solve.
Do not output anything else but the code block.

Requirements:
- Output only a single Python code block.
- Include all necessary imports explicitly (e.g., from typing import List, Optional, etc.).
- Do not assume any helper functions exist — define everything you use.
- The solution must be self-contained and executable.

Example 1:
Problem:
def add(a, b):
    \"\"\"Return the sum of a and b.\"\"\"

Code:
```python
def add(a, b):
    \"\"\"Return the sum of a and b.\"\"\"
    return a + b
```

Example 2:
Problem:
def is_palindrome(s: str) -> bool:
    \"\"\"Check if a string is a palindrome.\"\"\"

Code:
```python
def is_palindrome(s: str) -> bool:
    \"\"\"Check if a string is a palindrome.\"\"\"
    return s == s[::-1]
```

Problem:
{prompt}

Code:
"""

ALL_TEMPLATES = {"zero_shot": ZERO_SHOT_PROMPT, "few_shot": FEW_SHOT_PROMPT}
PROMPT_TEMPLATES = {k: ALL_TEMPLATES[k] for k in RUN_TEMPLATES}

## 3. Helper Functions (giữ nguyên)

In [ ]:
def run_eval(code, test_code, queue):
    local_scope = {}
    try:
        exec(code, local_scope)
        exec(test_code, local_scope)
        queue.put(True)
    except BaseException:
        queue.put(False)

def exec_code(code, test_code, timeout=5):
    """Executes the code and the test, returning True if passes, False otherwise (sandboxed in a subprocess)."""
    queue = multiprocessing.Queue()
    p = multiprocessing.Process(target=run_eval, args=(code, test_code, queue))
    p.start()
    p.join(timeout)

    if p.is_alive():
        p.terminate()
        p.join()
        return False

    return queue.get() if not queue.empty() else False

def calculate_pass_at_k(n, c, k):
    if n - c < k:
        return 1.0
    if c == 0:
        return 0.0
    try:
        return 1.0 - (math.comb(n - c, k) / math.comb(n, k))
    except ValueError:
        return 0.0

def clean_code(generated_text):
    match = re.search(r"```python\s*(.*?)\s*```", generated_text, re.DOTALL)
    if match:
        return match.group(1)
    match = re.search(r"```\s*(.*?)\s*```", generated_text, re.DOTALL)
    if match:
        return match.group(1)
    # Model đôi khi mở ```python mà không đóng (bị cắt ở max_new_tokens)
    match = re.search(r"```(?:python)?\s*(.*)", generated_text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return generated_text.strip()

def prompt_imports(prompt):
    """Các dòng import ở đầu prompt (vd `from typing import List`). Model hay chép chữ ký hàm
    nhưng bỏ import -> NameError dù logic đúng; cách eval chuẩn của HumanEval giữ lại prompt này."""
    return "\n".join(l for l in prompt.splitlines() if re.match(r"^(import |from \S+ import )", l))

def save_results(results, filename):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=4)

## 4. Generator + loader (mỗi lần nạp một model, giải phóng trước khi nạp model tiếp theo)

In [ ]:
from transformers import StoppingCriteria, StoppingCriteriaList

class StopOnCodeBlockEnd(StoppingCriteria):
    """Dừng ngay khi khối ``` đầu tiên được đóng. clean_code chỉ lấy khối đầu nên không đổi kết quả,
    nhưng bỏ qua phần giải thích dài mà model học từ OpenCodeInstruct (tiết kiệm tới ~10x thời gian)."""
    def __init__(self, tokenizer, prompt_len):
        self.tokenizer, self.prompt_len = tokenizer, prompt_len

    def __call__(self, input_ids, scores, **kwargs):
        # Trả về cờ dừng cho TỪNG chuỗi trong batch (num_return_sequences > 1)
        texts = self.tokenizer.batch_decode(input_ids[:, self.prompt_len:], skip_special_tokens=True)
        done = []
        for t in texts:
            first = t.find("```")
            done.append(first != -1 and t.find("```", first + 3) != -1)
        return torch.tensor(done, device=input_ids.device)


class QwenHFGenerator:
    def __init__(self, model, tokenizer, prompt_format="chatml"):
        self.model = model
        self.tokenizer = tokenizer
        self.prompt_format = prompt_format

    def _build_inputs(self, full_prompt):
        if self.prompt_format == "chatml":
            # Đúng format lúc train (không system prompt)
            full_prompt = f"<|im_start|>user\n{full_prompt}<|im_end|>\n<|im_start|>assistant\n"
        return self.tokenizer(full_prompt, return_tensors="pt").to(self.model.device)

    def generate(self, prompt_template, task_prompt, num_samples=1):
        full_prompt = prompt_template.format(prompt=task_prompt)
        inputs = self._build_inputs(full_prompt)

        prompt_len = inputs["input_ids"].shape[1]
        stopping = StoppingCriteriaList([StopOnCodeBlockEnd(self.tokenizer, prompt_len)])

        def _gen(n):
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    temperature=TEMPERATURE,
                    top_p=TOP_P,
                    do_sample=True,
                    num_return_sequences=n,      # sinh n mẫu trong 1 lần gọi: nhanh hơn nhiều so với n lần riêng lẻ
                    eos_token_id=STOP_IDS,
                    stopping_criteria=stopping,
                    pad_token_id=self.tokenizer.eos_token_id,
                )
            return [self.tokenizer.decode(o[prompt_len:], skip_special_tokens=True) for o in outputs]

        try:
            return _gen(num_samples)
        except Exception as e:                   # vd OOM -> thử lại từng mẫu một
            print(f"Batched generation failed ({e}); falling back to one-by-one")
            torch.cuda.empty_cache()
            samples = []
            for _ in range(num_samples):
                try:
                    samples.extend(_gen(1))
                except Exception as e2:
                    print(f"Error during generation: {e2}")
                    samples.append("")
            return samples


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

_loaded = {"name": None, "model": None, "tokenizer": None}

def load_model(name):
    """Giữ tối đa một model 7B trong VRAM; combo liền kề cùng model thì dùng lại."""
    if _loaded["name"] == name:
        return _loaded["model"], _loaded["tokenizer"]
    _loaded.update(model=None, tokenizer=None, name=None)
    gc.collect(); torch.cuda.empty_cache()
    mid = MODELS[name]
    tok = AutoTokenizer.from_pretrained(mid)
    mdl = AutoModelForCausalLM.from_pretrained(mid, quantization_config=bnb_config, device_map="auto").eval()
    _loaded.update(name=name, model=mdl, tokenizer=tok)
    print(f"Loaded {mid} | GPU mem: {torch.cuda.memory_allocated()/1e9:.1f} GB")
    return mdl, tok

## 5. Evaluation Loop

In [ ]:
def evaluate_dataset(dataset, dataset_key, template_name, prompt_template):
    print(f"\n===== Model [{MODEL_KEY}] | Dataset [{dataset_key}] | Template [{template_name}] | Format [{PROMPT_FORMAT}] =====")
    results = []
    total_pass_at_k = 0

    for item in tqdm(dataset):
        entry_point = item["entry_point"]

        start_time = time.time()
        raw_samples = generator.generate(prompt_template, item["prompt"], num_samples=NUM_SAMPLES)
        time_taken = time.time() - start_time

        correct_count = 0
        cleaned_codes = []
        for raw_text in raw_samples:
            code = clean_code(raw_text)
            full_code = prompt_imports(item["prompt"]) + "\n" + code
            eval_code = full_code + "\n" + item["test"] + f"\ncheck({entry_point})\n"
            passed = exec_code(full_code, eval_code, timeout=EXEC_TIMEOUT)
            correct_count += int(passed)
            cleaned_codes.append({"raw": raw_text, "extracted": code, "passed": passed})

        pass_at_k = calculate_pass_at_k(NUM_SAMPLES, correct_count, K_VALUE)
        total_pass_at_k += pass_at_k
        results.append({
            "task_id": item["task_id"],
            "task_name": entry_point,
            "samples": cleaned_codes,
            "correct_count": correct_count,
            "n": NUM_SAMPLES,
            "pass@k": pass_at_k,
            "time_taken": time_taken,
        })

    avg = total_pass_at_k / len(dataset)
    avg_time = sum(r["time_taken"] for r in results) / len(results)
    print(f"[{MODEL_KEY}][{dataset_key}][{template_name}] Avg pass@{K_VALUE}: {avg:.4f} - Avg time: {avg_time:.2f}s/it")

    filename = os.path.join(RESULTS_DIR, f"results_{MODEL_KEY}_{dataset_key}_{template_name}{RESULT_SUFFIX}.json")
    save_results(results, filename)
    print(f"Results saved to {filename}")
    return avg


summary = {}
for model_name, fmt in COMBOS:
    MODEL_KEY = model_key(model_name, fmt)
    PROMPT_FORMAT = fmt
    print(f"\n########## {MODELS[model_name]} | format={fmt} -> {MODEL_KEY} ##########")

    pending = []
    for dataset_key in DATASETS:
        for template_name in PROMPT_TEMPLATES:
            filename = os.path.join(RESULTS_DIR, f"results_{MODEL_KEY}_{dataset_key}_{template_name}{RESULT_SUFFIX}.json")
            pending.append((dataset_key, template_name, filename))

    for dataset_key, template_name, filename in pending:
        hf_path, split = DATASETS[dataset_key]
        dataset = load_dataset(hf_path, split=split)
        if LIMIT:
            dataset = dataset.select(range(LIMIT))
        run_name = f"{MODEL_KEY}/{dataset_key}/{template_name}"
        if SKIP_IF_EXISTS and not LIMIT and os.path.exists(filename):
            prev = json.load(open(filename, encoding="utf-8"))
            if len(prev) == len(dataset):
                summary[run_name] = sum(r["pass@k"] for r in prev) / len(prev)
                print(f"Skip {run_name} -> {summary[run_name]:.4f}")
                continue
        model, tokenizer = load_model(model_name)
        STOP_IDS = [tokenizer.convert_tokens_to_ids("<|im_end|>"), tokenizer.eos_token_id]
        generator = QwenHFGenerator(model, tokenizer, prompt_format=PROMPT_FORMAT)
        summary[run_name] = evaluate_dataset(dataset, dataset_key, template_name, PROMPT_TEMPLATES[template_name])

## 6. Summary & so sánh với FT

In [ ]:
save_results(summary, os.path.join(RESULTS_DIR, f"summary_control{RESULT_SUFFIX}.json"))

for ds in DATASETS:
    print(f"\n=== {ds} (few-shot, n={NUM_SAMPLES}) ===")
    row = {}
    for model_name, fmt in COMBOS:
        v = summary.get(f"{model_key(model_name, fmt)}/{ds}/few_shot")
        if v is not None:
            row[f"{model_name}/{fmt}"] = v
    row.update(FT_RESULTS[ds])
    for k, v in row.items():
        print(f"{k:28s} {v:.4f}")

print("\nĐọc kết quả:")
print("- base/raw (n=5) vs 0.7683/0.6951 (n=1): chênh bao nhiêu là nhiễu do n=1.")
print("- base/chatml vs base/raw: hiệu ứng riêng của prompt ChatML.")
print("- ft vs base/chatml: hiệu ứng thật của fine-tune (cùng prompt, cùng n).")